# 00 - Setup & smoke test
Run this first in your Databricks Free Edition workspace (serverless compute).
1. Installs `pytest` into this notebook's environment
2. Puts the repo root on `sys.path` so `import dsa`, `spark_lab`, `warehouse` work
3. Runs the reference test suite (should be all green)
4. Creates the schema the Spark labs write to

In [ ]:
%pip install -q pytest

### Put the repo on the Python path
Notebooks run from the `notebooks/` folder, so the repo root is one level up. Adding it to `sys.path` makes `import dsa`, `spark_lab`, `levels` and `warehouse` work. The output lists the repo's top-level folders so you can confirm it found the right place.

In [ ]:
import os, sys, subprocess

# Notebooks live in <repo>/notebooks, and the working directory is the notebook's folder.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print(sorted(os.listdir(REPO_ROOT)))

### Run the reference test suite
`run_tests` runs pytest in a **separate process**, so it always picks up your latest file edits. With `reference` it grades the worked solutions in `dsa/`, so every test should pass.

In [ ]:
def run_tests(target="reference", topic=None):
    """Run pytest in a fresh subprocess so edits to your files are always picked up."""
    env = {**os.environ, "DSA_TARGET": target, "PYTHONDONTWRITEBYTECODE": "1"}
    path = f"tests/test_{topic}.py" if topic else "tests"
    r = subprocess.run([sys.executable, "-m", "pytest", path, "-q", "-p", "no:cacheprovider", "--tb=line"],
                       cwd=REPO_ROOT, env=env, capture_output=True, text=True)
    print(r.stdout[-6000:] or r.stderr[-3000:])

run_tests("reference")

### Create the lab schema
The Spark labs write Delta tables to `catalog.schema`. These widgets appear at the top of the notebook, and the defaults (`workspace.interview_prep`) work on Free Edition. The cell creates the schema if needed and lists any tables already in it.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "interview_prep")
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("Spark version:", spark.version)
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))

**Setup complete.** If the tests passed and the schema is listed, go to `10_junior_walkthrough`.